# Tune an IMSTE embedding on MNIST

The default search balances 5-fold k-NN accuracy (60%) and robust spatial coverage (40%). It fits each trial for the estimator's default 200 epochs. Each trial embeds all selected feature rows; labels are used only for accuracy scoring. Change `SAMPLES_PER_CLASS` to adjust the balanced MNIST sample (default: 2,000 per digit).

Install the optional optimizer with `python -m pip install 'imste[tuning]'` if needed.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

repo_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'notebooks' / 'src' / 'digits_sweep.py').is_file()
)
sys.path.insert(0, str(repo_root / 'notebooks' / 'src'))
sys.path.insert(0, str(repo_root))

from digits_sweep import (
    load_mnist_per_class_data,
    make_live_tuning_display,
    plot_mnist_embedding,
)
from imste import IteratedMinimumSpanningTreeEmbedder

## Select and prepare the data

Set `SAMPLES_PER_CLASS` and rerun this cell to use a different number of images per digit. Sampling is balanced and deterministic.

In [2]:
SAMPLES_PER_CLASS = 2000  # Change this value to sample a different number per digit.
RANDOM_STATE = 42

X, y, data_info = load_mnist_per_class_data(
    samples_per_class=SAMPLES_PER_CLASS, random_state=RANDOM_STATE
)
print(
    f"Loaded {len(X):,} images from {data_info['source']} ({SAMPLES_PER_CLASS} per digit; {X.shape[1]} pixel features)."
)
display(data_info['class_counts'].to_frame())

Loaded 20,000 images from /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/mnist/cache/mnist_784_v1.npz (2000 per digit; 784 pixel features).


,images
0,2000
1,2000
2,2000
3,2000
4,2000
5,2000
6,2000
7,2000
8,2000
9,2000


## Set up the search

The `.tune()` defaults search `n_msts` (3–30), `learning_rate` (0.01–0.1), `negative_ratio` (1–10), `gravity_strength` (0.0–1.0), and `mean_shift_iterations` (0–5). They run 40 total trials, with 20 initial Latin hypercube samples, `k=5`, and coverage weight 0.4. `mapper` uses the estimator default of 200 epochs per fit.

In [ ]:
mapper = IteratedMinimumSpanningTreeEmbedder(
    random_state=RANDOM_STATE,
    device='auto',
)

## Watch intermediate results

The table reports accuracy, spatial coverage, and the weighted objective after every trial. The plot updates when a trial improves the combined objective.

In [ ]:
show_trial, trial_rows = make_live_tuning_display(y)

started = time.perf_counter()
search = mapper.tune(
    X, y,
    trial_callback=show_trial,
)
elapsed = time.perf_counter() - started
print(f'Search and final refit completed in {elapsed / 60:.1f} minutes')

## Best result

The estimator is refitted on all selected images using the configuration with the highest weighted objective. The returned result retains the separate accuracy, coverage, and combined scores.

In [ ]:
print(f"Best 5-fold {search.k}-NN accuracy: {search.best_accuracy:.3f}")
print(f"Best spatial coverage: {search.best_coverage:.3f}")
print(f"Best combined objective: {search.best_objective:.3f}")
display(pd.Series(search.best_params, name='best value').to_frame())
plot_mnist_embedding(
    mapper.embedding_, y,
    title=(
        f'Final refit: accuracy = {search.best_accuracy:.3f}; '        f'coverage = {search.best_coverage:.3f}'
    ),
)